# Deezer vs Rekordbox — Sync Check

Ce notebook compare les playlists Deezer `W - ...` avec les tags Rekordbox et produit un rapport de synchronisation.
Il s'appuie sur `server/deezer/extractor.py` et `server/deezer/sync_checker.py`.

## Initialisation

Connexion à l'API Deezer et à la base Rekordbox locale.

In [160]:
import sys
sys.path.insert(0, "..")

from server.deezer.extractor import DeezerExtractor
from server.deezer.sync_checker import check_sync
from worker.rekordbox.extractor import RekordboxExtractor

deezer = DeezerExtractor("656772321")
rb = RekordboxExtractor()

## Extraction Deezer

Récupère toutes les playlists publiques préfixées `W -` et leurs tracks via l'API Deezer (avec pagination).

In [161]:
# --- Extract Deezer ---
deezer_playlists = deezer.get_all_tracks()
print(f"{len(deezer_playlists)} playlists Deezer chargées")
for name, tracks in deezer_playlists.items():
    print(f"  {name}  ({len(tracks)} tracks)")

15 playlists Deezer chargées
  W - Classic/Min. Techno  (31 tracks)
  W - Deep House  (43 tracks)
  W - Downtempo  (13 tracks)
  W - Electro Brut  (9 tracks)
  W - French Touch  (46 tracks)
  W - Hard/Dark Techno  (38 tracks)
  W - Hard Groove  (0 tracks)
  W - Melodic Techno  (27 tracks)
  W - Misc. Tracks  (12 tracks)
  W - Nu Disco  (39 tracks)
  W - Psytrance  (40 tracks)
  W - Tech House  (53 tracks)
  W - Trance Techno  (156 tracks)
  W - UK Garage  (24 tracks)
  W - UK House  (64 tracks)


## Extraction Rekordbox

Récupère tous les tags MyTag de style et leurs tracks depuis la base Rekordbox locale.

In [162]:
# --- Extract Rekordbox ---
rb_tags = rb.get_tags_structure()
rb_by_tag = {}
for group, tags in rb_tags.items():
    for tag in tags:
        tracks = rb.get_tracks_by_tag(tag)
        rb_by_tag[tag] = [rb.get_track_metadata(t) for t in tracks]

print(f"{len(rb_by_tag)} tags Rekordbox chargés")
for tag, tracks in rb_by_tag.items():
    print(f"  {tag}  ({len(tracks)} tracks)")

{}
24 tags Rekordbox chargés
  IMPORTED  (13 tracks)
  Deep House  (44 tracks)
  TO_SORT  (2 tracks)
  TO_ENERGY  (647 tracks)
  TO_RATE  (208 tracks)
  TO_CUE  (625 tracks)
  TO_LOOP  (625 tracks)
  TO_COLOR  (647 tracks)
  Downtempo  (13 tracks)
  French Touch  (46 tracks)
  Electro Brut  (9 tracks)
  UK House  (65 tracks)
  Tech House  (53 tracks)
  Melodic Techno  (27 tracks)
  Classic/Min. Techno  (31 tracks)
  UK Garage  (29 tracks)
  Hard/Dark Techno  (38 tracks)
  Psytrance  (41 tracks)
  Trance Techno  (164 tracks)
  Nu-Disco  (39 tracks)
  Misc. Tracks  (12 tracks)
  SoundCloud  (16 tracks)
  Mix  (24 tracks)
  Mix_001  (52 tracks)


## Rapport de synchronisation

Compare les deux sources par titre normalisé et émet des flags :
- `DOWNLOAD_NEEDED` — sur Deezer, absent de RB → télécharger via Deemix
- `MISPLACED` — présent des deux côtés mais dans des catégories différentes
- `ANOMALY` — dans RB, absent de Deezer → à investiguer
- `INFO` — potentiellement indispo sur Deezer

In [163]:
report = check_sync(deezer_playlists, rb_by_tag)
print(report.summary())


[DOWNLOAD_NEEDED] (1)
  - Collect 200 — Pull Up  (playlist: W - UK House)
